# 03 What to measure (blind)

**Task.** Follow the Procedure of Section 3 on a **pilot**: measure every candidate channel for 2 years, certify
which (excitation, channel) pairs carry influence (Corollary 4), predict from the pilot how each plan would do if
it ran longer (eq. 7), and choose a plan. Then check the prediction against what the longer data report.

**Reads:** `data/measurements/` (the first 2 years as the pilot), `data/sketch/`. **Never reads:** `data/truth/`.

**How to read the tags.** A code cell that uses a result of the paper starts with a comment such as
`# [Thm 2] threshold c_alpha`. The tag names the result; the rest of the line says what the cell does. Numbering
follows the paper: Definition 1 and Lemma 1 (first lags), Theorem 2 (certified links), Proposition 3 (lag scan),
Corollary 4 (channel certificates), Algorithm 1 and the Procedure of Section 3, equations (3) (impulse response),
(4) (residuals), (5) (the statistic), (6) (the threshold), (7) (power) and (8) (measurement duration), and the
conditions C1 (measured innovations) and C2 (predictable regressors and weights) of Theorem 2.

In [1]:
import os, sys, glob, pickle, warnings, numpy as np, pandas as pd
warnings.filterwarnings('ignore')            # scikit-learn convergence messages from the Lasso baseline
sys.path.insert(0, 'lib')
import certify as K, farmlib as FL
np.set_printoptions(precision=2, suppress=True, linewidth=150)
files = sorted(glob.glob('data/measurements/*.npz'))
SK = FL.load_sketch('data/sketch/channel_sketch.npz')
TAU, TAU0 = FL.lags(SK), FL.tau0(SK)          # lags tau_{d,o} and tau^0_c, from the sketch alone
ALPHA = 0.1
from scipy.stats import norm
PILOT_YEARS = 2

## Step 1 of the Procedure: pilot, then channel certificates (Corollary 4)
For each candidate pair $(d,o)$, the lag scan runs over all farm pairs at level $\alpha/(\bar L|\mathcal D||\mathcal O|)$.
The pair is certified if any farm pair is reported at any lag. The probability of certifying any pair that is
**not reached** is at most $\alpha$, with no failure term.

In [2]:
# [Cor 4] channel certificates from a 2-year pilot in which every candidate channel is measured
cert = []
for f in files:
    R = FL.load_measurements(f)
    Rp = FL.view(R, PILOT_YEARS * 365, FL.CANDIDATES, [0, 1])
    cert.append(K.channel_certificates(Rp, FL.CANDIDATES, [0, 1], FL.SHARED, ALPHA))
rows = []
for (d, o) in cert[0]:
    found = [c[(d, o)] for c in cert if c[(d, o)] is not None]
    rows.append(dict(excitation=FL.EXCITATIONS[d], channel=FL.NAMES[o],
                     share_certified=np.mean([c[(d, o)] is not None for c in cert]),
                     lag_found=sorted(set(found)) if found else '-', sketch_lag=TAU.get((d, o), 'not reached')))
pd.DataFrame(rows)

,excitation,channel,share_certified,lag_found,sketch_lag
0,canal-turn deviation,soil moisture,0.0,-,4
1,canal-turn deviation,GW level,0.0,-,4
2,canal-turn deviation,belief survey,0.0,-,3
3,canal-turn deviation,pumping,0.0,-,3
4,canal-turn deviation,spraying,0.0,-,not reached
5,canal-turn deviation,canopy temp. (decoy),0.0,-,not reached
6,canal-turn deviation,price survey (decoy),0.0,-,not reached
7,pest-pressure innovation,soil moisture,0.0,-,not reached
8,pest-pressure innovation,GW level,0.0,-,not reached
9,pest-pressure innovation,belief survey,0.0,-,not reached


Read this as Corollary 4 says: a certified pair is **useful**; a pair that is not certified **may still carry
influence** that the pilot was too short to show. A pair can be certified useful but never useless. The decoys and
every pair that the sketch says is not reached should (almost) never be certified.

## Step 2: predict each plan from the pilot (eq. 7)
For each farm pair, the pilot gives $Z^{\rm pilot}_{ij}$. By eq. (7) the mean of $Z$ grows like $\sqrt{T}$, so a plan
that runs for $T$ days is predicted to give $\hat\mu_{ij}(T)=|Z^{\rm pilot}_{ij}|\sqrt{T/T_{\rm pilot}}$, and to report the
pair with probability $\Pr(|\mathcal N(\hat\mu,1)|\ge c_\alpha)$, combined over excitations. Summing over pairs
predicts how many links the plan will report.

*Limit.* This uses only the pilot. It cannot predict links that are invisible in the pilot, and it cannot separate
the link weight from the gain, so it cannot give the paper's "years to find 90% of links" (Table 3). Notebook 04
checks eq. (7) with the true gains.

In [3]:
# [eq. (7), scaled from the pilot] predicted number of reports after longer durations, for each plan
PLANS_CMP = ['Spray+Pest', 'Pump+Spray', 'Pump', 'SM+Pump', 'SM', 'Survey', 'Decoys']
HORIZONS = [3, 5]
pred = []
for f in files:
    R = FL.load_measurements(f)
    for plan_name in PLANS_CMP:
        p_ = FL.plan(plan_name)
        Rp = FL.view(R, PILOT_YEARS * 365, p_['outputs'], p_['excitations'], p_['shared'], p_.get('farms_out'))
        W_hat, Zs, infos = K.certify_plan(Rp, p_, TAU, ALPHA, tau0=TAU0)
        row = dict(community=os.path.splitext(os.path.basename(f))[0][-2:], plan=plan_name, pilot_reports=int(W_hat.sum()))
        for Hy in HORIZONS:
            q = None
            for d, Zd in Zs.items():
                mu = np.abs(np.nan_to_num(Zd)) * np.sqrt(Hy / PILOT_YEARS)
                c = infos[d]['c_alpha']
                pr = norm.cdf(mu - c) + norm.cdf(-mu - c); np.fill_diagonal(pr, 0)
                q = (1 - pr) if q is None else q * (1 - pr)
            row[f'pred_{Hy}y'] = float((1 - q).sum()) if q is not None else 0.0
            # what Hy years of data actually report (still blind: counts only)
            Rh = FL.view(R, Hy * 365, p_['outputs'], p_['excitations'], p_['shared'], p_.get('farms_out'))
            row[f'obs_{Hy}y'] = int(K.certify_plan(Rh, p_, TAU, ALPHA, tau0=TAU0)[0].sum())
        pred.append(row)
P = pd.DataFrame(pred)
P.groupby('plan')[['pilot_reports'] + [f'{a}_{h}y' for h in HORIZONS for a in ('pred', 'obs')]].mean().round(1).loc[PLANS_CMP]

,pilot_reports,pred_3y,obs_3y,pred_5y,obs_5y
plan,,,,,
Spray+Pest,13.3,16.8,18.0,19.3,20.0
Pump+Spray,12.4,16.0,17.0,19.1,19.9
Pump,0.0,0.9,0.3,3.2,2.1
SM+Pump,0.0,0.3,0.0,1.2,1.5
SM,0.0,0.1,0.0,0.4,0.0
Survey,0.0,0.0,0.0,0.0,0.0
Decoys,0.0,0.0,0.0,0.0,0.0


In [4]:
# [Procedure, step 3] choose the plan predicted to report the most links within the budget
BUDGET_YEARS = 3
best = P.groupby('plan')[f'pred_{BUDGET_YEARS}y'].mean().sort_values(ascending=False)
print(f'predicted links reported after {BUDGET_YEARS} years, per community:'); print(best.round(1))
print('\nchosen plan:', best.index[0])

predicted links reported after 3 years, per community:
plan
Spray+Pest    16.8
Pump+Spray    16.0
Pump           0.9
SM+Pump        0.3
SM             0.1
Survey         0.0
Decoys         0.0
Name: pred_3y, dtype: float64

chosen plan: Spray+Pest


The pilot alone recommends measuring **spraying with pest scouting** (plus the shared series). It does
**not** certify soil moisture, groundwater level, the survey or the decoys, and it predicts that they would report
almost nothing within 5 years. That is not a certificate that they are useless (Corollary 4 never gives one); it is
a prediction that they are not worth the cost, which is the paper's conclusion in Section 4.

The predicted counts track what the longer data report. For weak plans (Pump) the pilot-scaled prediction runs
high, because $|Z^{\rm pilot}|$ overstates a small $\mu$: noise only adds to an absolute value.